In [1]:
import math

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import matplotlib.pyplot as plt

print(torch.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

2.11.0
Using device: mps


In [2]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("python-dotenv not installed, reading HF_TOKEN from environment only")

hf_token = os.getenv("HF_TOKEN")

python-dotenv not installed, reading HF_TOKEN from environment only


# Create Dataset

In [3]:
from datasets import load_from_disk
import pandas as pd

ds = load_from_disk("./snli-ve")

Loading dataset from disk:   0%|          | 0/147 [00:00<?, ?it/s]

# Using Dataloader loading data from dataset- image-text

In [4]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# train_transform = transforms.Compose([
#     transforms.RandomResizedCrop(224),
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
# ])

def preprocess(examples):
    inputs = tokenizer(
        examples["hypothesis"],
        truncation=True,
        max_length=128
    )
    inputs["pixel_values"] = [
        image_transform(img.convert("RGB"))
        for img in examples["image"]
    ]
    inputs["labels"] = examples["label"]
    return inputs

def get_cleaned_dataset(set_name):
    split = ds[set_name].select(range(1000))
    split = split.filter(lambda label: label != -1, input_columns=["label"])
    split.set_transform(preprocess)
    return split

train_ds = get_cleaned_dataset("train")
val_ds = get_cleaned_dataset("validation")


text_collator = DataCollatorWithPadding(tokenizer)

def multimodal_collate_fn(features):
    images = [f.pop("pixel_values") for f in features]
    batch = text_collator(features)          # pad input_ids / attention_mask
    batch["pixel_values"] = torch.stack(images)
    return batch

# macOS + Jupyter 下 num_workers>0 容易因 multiprocessing 报错，本地调试用 0
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=multimodal_collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=multimodal_collate_fn,
    num_workers=0
)

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
batch = next(iter(train_loader))

print(batch.keys())
for k, v in batch.items():
    print(k, type(v), getattr(v, "shape", None), getattr(v, "dtype", None))

# Custom CAMC-NET Model

In [5]:
import torch.nn as nn
from transformers import ViTModel, BertModel

class ImageEncoder(nn.Module):
    def __init__(self, model="google/vit-base-patch16-224"):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model)

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values=pixel_values)
       
        img_emb = outputs.last_hidden_state   # [B, 1+N_patch, 768]
        return img_emb


class TextEncoder(nn.Module):
    def __init__(self, model="bert-base-uncased"):
        super().__init__()
        self.bert = BertModel.from_pretrained(model)

    def forward(self, input_ids, attention_mask=None):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        txt_emb = output.last_hidden_state    # [B, N_txt, 768]
        return txt_emb

In [6]:
class ContradictionAwareLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm_t1 = nn.LayerNorm(hidden_dim)
        self.ffn_t = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )
        self.norm_t2 = nn.LayerNorm(hidden_dim)

    def forward(self, img_emb, txt_emb, img_mask=None):
       
        cross_out, attn_weights = self.cross_attn(
            query=txt_emb,
            key=img_emb,
            value=img_emb,
            key_padding_mask=img_mask
        )  
        txt_emb = self.norm_t1(txt_emb + cross_out)

        ffn_out = self.ffn_t(txt_emb)
        txt_emb = self.norm_t2(txt_emb + ffn_out)

        return txt_emb, attn_weights

In [7]:
class ContradictionAwareEncoder(nn.Module):
    def __init__(self, hidden_dim, num_heads=8, num_layers=6):
        super().__init__()
        self.img_proj = nn.Linear(hidden_dim, hidden_dim)
        self.layers = nn.ModuleList([
            ContradictionAwareLayer(hidden_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, img_emb, txt_emb):
        img_emb = self.img_proj(img_emb)   # ablation: img_proj

        all_attn_weights = []
        for layer in self.layers:
            txt_emb, attn_weight = layer(img_emb, txt_emb)
            all_attn_weights.append(attn_weight)   

        text_pool = txt_emb[:, 0]   
        img_pool = img_emb[:, 0]    

        fusion_feature = torch.cat([
            img_pool,
            text_pool,
            torch.abs(text_pool - img_pool),
            text_pool * img_pool
        ], dim=-1)                  # [B, 4*hidden]

        return fusion_feature, all_attn_weights

In [8]:
class CAMC(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8,
                 num_layers=6, num_classes=3):
        super().__init__()
        self.ie = ImageEncoder()
        self.te = TextEncoder()
        self.encoder = ContradictionAwareEncoder(
            hidden_dim, num_heads, num_layers
        )


        for p in self.ie.parameters():
            p.requires_grad = False
        for p in self.te.parameters():
            p.requires_grad = False
        self.ie.eval()
        self.te.eval()

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, pixel_values, input_ids, attention_mask=None):
        with torch.no_grad():  
            img_emb = self.ie(pixel_values)
            txt_emb = self.te(input_ids, attention_mask)

        fusion_feature, all_attn_weights = self.encoder(img_emb, txt_emb)
        logits = self.classifier(fusion_feature)

        return logits, all_attn_weights

    def train(self, mode=True):
        super().train(mode)   
        self.ie.eval()        
        self.te.eval()
        return self

# Set model, optimizer, loss fn, 

In [9]:
model = CAMC(hidden_dim=768).to(device) 
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Training

In [ ]:
import os
import random
import json
import time

ROOT = './'
CPPath = ROOT + 'checkpoints/'
ResultPath = ROOT + 'results/'
os.makedirs(CPPath, exist_ok=True)
os.makedirs(ResultPath, exist_ok=True)

def train(model, num_epochs):
    model = model.to(device)
    train_loss_list = []
    train_acc_list = []
    val_loss_list = []
    val_acc_list = []

    seed = 0
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

    best_val_acc = 0.0
    best_val_loss = float('inf')
    best_epoch = 0

    # 每个 epoch 实时追加写 log，训练中断也有记录；文件名带时间戳，多次运行不覆盖
    run_name = time.strftime("%Y%m%d-%H%M%S")
    log_path = ResultPath + f'train_{run_name}.log'
    lr = optimizer.param_groups[0]['lr']
    with open(log_path, 'a') as f:
        f.write(f"run: {run_name} | device: {device} | epochs: {num_epochs} | "
                f"lr: {lr} | seed: {seed} | "
                f"train/val size: {len(train_ds)}/{len(val_ds)}\n")

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0
        num_train_samples = 0

        for batch in train_loader:
            img_feature = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits, _ = model(img_feature, input_ids, attention_mask)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(logits, 1)

            running_loss += loss.item()
            running_correct += (predicted == labels).sum().item()
            num_train_samples += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = running_correct / num_train_samples

        # validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        num_val_samples = 0
        with torch.no_grad():
            for batch in val_loader:
                img_feature = batch["pixel_values"].to(device)
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits, _ = model(img_feature, input_ids, attention_mask)
                loss = criterion(logits, labels)

                _, predicted = torch.max(logits, 1)
                val_loss += loss.item()
                val_correct += (predicted == labels).sum().item()
                num_val_samples += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = val_correct / num_val_samples

        # save checkpoint while best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), CPPath + 'best.pt')

        # record for plot
        train_loss_list.append(train_loss)
        train_acc_list.append(train_acc)
        val_loss_list.append(val_loss)
        val_acc_list.append(val_acc)

        log_line = (f"Epoch {epoch+1}/{num_epochs} | "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Train Acc: {train_acc:.4f} | "
                    f"Val Loss: {val_loss:.4f} | "
                    f"Val Acc: {val_acc:.4f} | "
                    f"Best Val Acc: {best_val_acc:.4f}")
        print(log_line)
        with open(log_path, 'a') as f:
            f.write(log_line + '\n')

    metrics = {
        "train_loss": train_loss_list,
        "train_acc": train_acc_list,
        "val_loss": val_loss_list,
        "val_acc": val_acc_list,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
    }
    with open(ResultPath + f'metrics_{run_name}.json', 'w') as f:
        json.dump(metrics, f, indent=2)

    return metrics

In [12]:
# 本地验证用小 epoch 数，HPC 上正式跑再调大
num_epochs = 2
metrics = train(model, num_epochs)

Epoch 1/2 | Train Loss: 1.1058 | Train Acc: 0.3110 | Val Loss: 1.1047 | Val Acc: 0.3090 | Best Val Acc: 0.3090
Epoch 2/2 | Train Loss: 1.1065 | Train Acc: 0.3320 | Val Loss: 1.1047 | Val Acc: 0.3090 | Best Val Acc: 0.3090


# Plot results

In [ ]:
# Your graph
def plot_loss_acc(metrics, num_epochs):
    epochs = range(1, num_epochs + 1)
    
    fig = plt.figure(figsize=(12,5))
    
    # Loss
    plt.subplot(1,2,1)
    plt.plot(epochs, metrics["train_loss"], label='Train Loss')
    plt.plot(epochs, metrics["val_loss"], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    
    # Accuracy
    plt.subplot(1,2,2)
    plt.plot(epochs, metrics["train_acc"], label='Train Accuracy')
    plt.plot(epochs, metrics["val_acc"], label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training vs Validation Accuracy')
    plt.legend()
    
    plt.tight_layout()
    fig.savefig(ResultPath + "training_loss.svg")
    plt.show()

plot_loss_acc(metrics, len(metrics["train_loss"]))

# Sanity Check

In [11]:

model = CAMC(hidden_dim=768).to(device)   

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable/1e6:.1f}M / total: {total/1e6:.1f}M")

model.train()
print("encoders stay eval:", not model.ie.training, not model.te.training)  # True True 才对

batch = next(iter(train_loader))
with torch.no_grad():
    logits, attn = model(
        batch["pixel_values"].to(device),
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device)
    )
print("logits:", logits.shape)                        # [32, 3]
print("attn layers:", len(attn), attn[0].shape)       # 6 layers, [32, N_txt, N_img]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable: 45.5M / total: 241.4M
encoders stay eval: True True
logits: torch.Size([32, 3])
attn layers: 6 torch.Size([32, 28, 197])
